In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD
from sklearn.feature_extraction.text import TfidfTransformer

In [2]:
DTCM = pd.read_csv("/kaggle/input/notebooks/isabellouisedelgado/text-as-data-final-project-bow-dctm-and-tfidf/GothicNovels_DTCM.csv")
LIB = pd.read_csv("/kaggle/input/notebooks/isabellouisedelgado/text-as-data-final-project-corpus/GothicNovels_LIB.csv")
VOCAB = pd.read_csv("/kaggle/input/notebooks/isabellouisedelgado/text-as-data-final-project-corpus/GothicNovels_VOCAB.csv")

In [3]:
tfidf_engine = TfidfTransformer()
TFIDF = pd.DataFrame(tfidf_engine.fit_transform(DTCM).toarray(), index=DTCM.index, columns=tfidf_engine.get_feature_names_out())
TFIDF

,book_id,chap_num,1,10,100,1018,1030,10th,11,1140,...,zoöphagy,zurich,à,æneis,æra,æt,ætat,æther,état,œdipus
0,0.965071,1.795481e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.147993,3.097532e-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.114165,2.655001e-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.111477e-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.351501,8.991893e-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.194746,5.434786e-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1.000000,2.477596e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.074418e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0
116,1.000000,2.973115e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
117,1.000000,3.468634e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
118,1.000000,3.964153e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
VOCAB = DTCM.sum().to_frame('n')
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = np.log2(1/VOCAB.p)
VOCAB['df'] = DTCM.astype(bool).sum()
VOCAB['dfidf'] = VOCAB.df * np.log2(len(DTCM)/VOCAB.df)

In [5]:
df_thresh = 50 # Remove proper nouns and one-offs
i_thresh = 10 # Remove stopwords
SIGS = VOCAB[(VOCAB.df >= df_thresh) & (VOCAB.i > i_thresh)].index
len(SIGS)

545

In [6]:
 TFIDF_SIGS = TFIDF[SIGS].copy()

## PCA DCM

In [7]:
from sklearn.decomposition import PCA

n_components = 5
pca_engine = PCA(n_components=n_components, random_state=42)
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF_SIGS), index=TFIDF_SIGS.index)
DCM

,0,1,2,3,4
0,-0.414806,-0.013358,-0.027738,0.020219,0.037062
1,0.424338,-0.067125,-0.128136,-0.036664,0.038166
2,0.361992,-0.153650,-0.060226,-0.050179,0.057774
3,0.159669,0.290882,-0.126859,0.132592,-0.049113
4,0.370909,-0.239405,-0.046300,-0.071458,0.026385
...,...,...,...,...,...
115,-0.441172,-0.000669,0.002276,-0.009984,-0.001529
116,-0.441190,-0.000671,0.002276,-0.009984,-0.001527
117,-0.441176,-0.000670,0.002275,-0.009983,-0.001526
118,-0.441184,-0.000668,0.002276,-0.009984,-0.001528


## PCA Loadings

In [8]:
COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_))
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index = TFIDF_SIGS.columns
COMPS.index.name = 'term_str'
LOADINGS = COMPS.T
LOADINGS

term_str,chap_num,a,able,about,above,account,across,act,added,afraid,...,work,world,would,years,yes,yet,you,young,your,yourself
PC0,0.009668,0.073263,0.001539,0.005589,0.001622,0.000717,0.000925,0.001331,0.001665,0.001294,...,0.002910,0.001650,0.009595,0.001199,0.003538,0.003484,0.039766,0.002828,0.008248,0.000984
PC1,0.001834,-0.001889,0.000526,0.001109,-0.001121,0.000422,0.000020,0.000091,-0.000745,0.000877,...,0.001712,0.000412,0.001413,-0.000328,-0.001593,0.001085,0.024561,0.000183,0.003221,0.000447
PC2,-0.007895,-0.008445,-0.000133,-0.001487,-0.000218,0.000258,-0.000247,-0.000044,-0.000643,0.000106,...,0.000986,0.000660,0.001897,0.000234,-0.001961,0.000810,-0.005070,0.001373,0.000535,0.000066
PC3,0.009333,0.002250,-0.000433,0.001095,0.000618,0.000192,-0.000229,0.000728,-0.000155,0.000103,...,-0.001963,0.000191,-0.000690,-0.000306,0.004263,-0.000738,0.042793,0.002941,0.005432,0.001029
PC4,0.011503,0.002034,0.000027,0.000344,-0.000094,-0.000121,0.000371,0.000377,0.000029,-0.000025,...,0.001498,0.000396,-0.000362,-0.000460,0.000986,0.000499,0.001088,-0.002214,-0.001276,-0.000232


## Top 5 positive terms for first component and Top 5 negative terms for second component

In [9]:
top_terms_sk = {}
data = []
for i in range(5):
    for j in [0, 1]:
        data.append((i, j, ' '.join(COMPS.sort_values(f'PC{i}', ascending=bool(j)).head().index.to_list())))
comp_strs = pd.DataFrame(data)
comp_strs.columns =  ['pc', 'pole', 'gloss']
comp_strs = comp_strs.set_index(['pc', 'pole'])
comp_strs.unstack()

gloss                                      
pole                      0                                     1
pc                                                               
0           the and to of i  although wished form account despair
1           i you me my and                    the of she had her
2          her she to he we                    i the a chap_num m
3     you m s what chap_num                        and i we as of
4      we chap_num he us is                       i her she my to

## PCA Visualization 1

In [10]:
DOC = pd.DataFrame(index=DTCM.index).join(LIB)

In [11]:
def vis_pcs(x, y, color_col):
    return px.scatter(DCM.join(DOC), x, y, 
                      color=color_col, 
                      hover_name='title',
                      marginal_x='box', 
                      height=1000)

In [12]:
def vis_loadings(COMPS, a=0, b=1, hover_name='term_str'):
    return px.scatter(COMPS.reset_index(), f"PC{a}", f"PC{b}", 
                      text='term_str', 
                      marginal_x='box', 
                      height=800)

In [13]:
vis_pcs(0, 1, 'period')

In [14]:
vis_loadings(COMPS, 0, 1)

## PCA Visualization 2

In [15]:
vis_pcs(2, 3, 'period')

In [16]:
vis_loadings(COMPS, 2, 3)